In [1]:
!pip install transformers

In [2]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [5]:
openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_API_KEY')
)
response = openrouter.chat.completions.create(
    model = "nvidia/nemotron-3.5-lightning:free",
    messages = [
        {"role": "user", "content": "Tell me a joke"},
    ]
    )
print(response.choices[0].message.content)

Why don't scientists trust atoms? Because they make up everything!


In [12]:
gpt_model = "dots-studio/dots-3-note-preview:free"
claude_model = "nvidia/nemotron-3.5-lightning:free"

In [19]:

from openai import OpenAI
from google.colab import userdata

# Connect to OpenRouter
openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY")
)


# Different personalities
gpt_system = """
You are GPT, a very argumentative chatbot.
You disagree with the other AI when you have a reason to,
challenge its assumptions, and respond in a snarky but intelligent way.
Keep your replies concise and stay on the topic.
"""

claude_system = """
You are Claude, a polite and courteous chatbot.
Try to find common ground with the other AI, calm disagreements,
and explain your reasoning thoughtfully.
Keep your replies concise and stay on the topic.
"""

# Conversation histories
gpt_messages = ["Hi there!"]
claude_messages = ["Hi!"]



def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]

    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})

    response = openrouter.chat.completions.create(
        model=gpt_model,
        messages=messages,
        max_tokens=2500,
        temperature=0.7,
        extra_body={
            "reasoning": {"effort": "low"}
        }
    )

    choice = response.choices[0]
    content = choice.message.content

    print("Finish reason:", choice.finish_reason)

    if not isinstance(content, str) or not content.strip():
        print("No final text was returned.")
        print("Reasoning tokens:",
              getattr(choice, "usage", None))
        return None

    return content.strip()




def call_claude():
    messages = [{"role": "system", "content": claude_system}]

    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({
            "role": "user",
            "content": gpt
        })
        messages.append({
            "role": "assistant",
            "content": claude_message
        })

    messages.append({
        "role": "user",
        "content": gpt_messages[-1]
    })

    response = openrouter.chat.completions.create(
        model=claude_model,
        messages=messages,
        max_tokens=1500,
        temperature=0.7
    )

    choice = response.choices[0]
    message = choice.message

    print("Finish reason:", choice.finish_reason)
    print("Usage:", response.usage)

    if not isinstance(message.content, str) or not message.content.strip():
        print("Empty content:", repr(message.content))
        print("Reasoning:", repr(getattr(message, "reasoning", None)))
        return None

    return message.content.strip()



# Let the models interact
for i in range(5):
    gpt_reply = call_gpt()
    gpt_messages.append(gpt_reply)

    print(f"\nGPT ({gpt_model}):\n{gpt_reply}")

    claude_reply = call_claude()
    claude_messages.append(claude_reply)

    print(f"\nCLAUDE-STYLE MODEL ({claude_model}):\n{claude_reply}")

print("\nConversation complete!")


Finish reason: stop

GPT (dots-studio/dots-3-note-preview:free):
Hey! What's on your mind?
Finish reason: stop
Usage: CompletionUsage(completion_tokens=281, prompt_tokens=84, total_tokens=365, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=256, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0})

CLAUDE-STYLE MODEL (nvidia/nemotron-3.5-lightning:free):
Not much! Just here and ready to help with whatever you'd like to chat about. How are you doing?
Finish reason: stop

GPT (dots-studio/dots-3-note-preview:free):
I'm doing great, thanks! I'm all ears—what's on your mind today?
Finish reason: stop
Usage: CompletionUsage(completion_tokens=418, prompt_tokens=139, total